# HydroSafe: Digital Twin Framework for Heavy Metal Water Contamination Detection

**Research title:** *A Digital Twin Framework for Deep Learning-Based Heavy Metal Water Contamination Detection Using Carbon Nanomaterial Sensor Response Modeling*

This notebook provides an end-to-end reproducible workflow for the HydroSafe study. It covers data loading, regulatory threshold mapping, cleaning, feature engineering, exploratory analysis, machine learning baselines, repeated cross-validation, deep learning models, digital twin decision logic, interpretability analysis, model artifact export, and manuscript-ready tables and figures.

The primary decision pipeline follows a concentration-driven regulatory logic:

```text
sensor-response features -> predicted metal -> predicted concentration -> metal-specific threshold -> contamination status
```

Direct status classification is reported as a baseline, whereas the regulatory digital twin pipeline is treated as the main interpretable decision workflow.

## 0. Runtime and GPU/CUDA Configuration

The notebook can be executed in Google Colab or a local Python environment. A GPU runtime is recommended for TensorFlow/Keras models. Tree-based tabular models are executed on CPU unless GPU-supported libraries are available.

Recommended Colab setting:

```text
Runtime -> Change runtime type -> Hardware accelerator -> GPU
```

In [ ]:
# ================================================================
# 0. ENVIRONMENT SETUP, GPU/CUDA CHECK, DRIVE OUTPUT CONFIGURATION
# ================================================================

import os, sys, json, time, shutil, subprocess, platform, warnings, math, random
from pathlib import Path
from datetime import datetime
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)

# Reproducible timestamp for this run
RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
PROJECT_NAME = 'digital_twin_heavy_metal_water_contamination_q1_revised'

# Mount Google Drive in Colab
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive') / f'{PROJECT_NAME}_{RUN_TIMESTAMP}'
else:
    BASE_DIR = Path.cwd() / f'{PROJECT_NAME}_{RUN_TIMESTAMP}'

# Folder structure for reproducible research outputs
DIRS = {
    'root': BASE_DIR,
    'raw': BASE_DIR / '01_data_raw',
    'processed': BASE_DIR / '02_data_processed',
    'tables': BASE_DIR / '03_tables',
    'figures': BASE_DIR / '04_figures',
    'models': BASE_DIR / '05_models',
    'reports': BASE_DIR / '06_reports_and_logs',
    'zip': BASE_DIR / '07_zip_export'
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print('Output directory:', BASE_DIR)

# GPU/CUDA diagnostics
print('\n=== Python and system ===')
print('Python:', sys.version)
print('Platform:', platform.platform())

print('\n=== NVIDIA GPU/CUDA check ===')
gpu_info = {'nvidia_smi_available': False, 'nvidia_smi_output': None}
try:
    out = subprocess.check_output(['nvidia-smi'], text=True)
    gpu_info['nvidia_smi_available'] = True
    gpu_info['nvidia_smi_output'] = out
    print(out)
except Exception as e:
    print('nvidia-smi not available or GPU not enabled:', e)

with open(DIRS['reports'] / 'gpu_cuda_diagnostics_raw.txt', 'w', encoding='utf-8') as f:
    f.write(gpu_info.get('nvidia_smi_output') or 'nvidia-smi not available')

In [ ]:
# ================================================================
# 1. INSTALL AND IMPORT PACKAGES - COLAB SAFE ENVIRONMENT FIX
#    Fixes: ValueError: numpy.dtype size changed / binary incompatibility.
# ================================================================

import os, sys, json, subprocess
from pathlib import Path

# IMPORTANT:
# Colab sometimes has mixed binary wheels after pip installs. This cell pins
# NumPy/Pandas/SciPy/Scikit-learn together, then restarts the runtime once.
# After the automatic restart/crash message, run the notebook again from the top.

ENV_MARKER = Path('/content/.dt_heavy_metal_env_ready_numpy_pandas_sklearn_v2')
FORCE_REBUILD_ENV = False  # change to True only if you want to reinstall packages again

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB and (FORCE_REBUILD_ENV or not ENV_MARKER.exists()):
    print('Preparing a clean Colab Python environment...')
    print('This is normal: after installation, the runtime will restart once. Then run all cells again.')
    
    packages = [
        'numpy==1.26.4',
        'pandas==2.2.2',
        'scipy==1.13.1',
        'scikit-learn==1.6.1',
        'joblib==1.4.2',
        'openpyxl==3.1.5',
        'xlsxwriter==3.2.0',
        'xgboost==2.1.4',
        'lightgbm==4.5.0',
        'catboost==1.2.7',
        'optuna==4.1.0',
        'shap==0.46.0'
    ]
    
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        '--upgrade', '--force-reinstall', '--no-cache-dir',
        *packages
    ])
    
    ENV_MARKER.write_text('ready', encoding='utf-8')
    print('\nEnvironment packages installed successfully.')
    print('Restarting runtime now to prevent NumPy/Pandas binary incompatibility...')
    os.kill(os.getpid(), 9)

# Imports happen only after the environment is consistent.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, RepeatedStratifiedKFold, KFold, RepeatedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, f1_score, classification_report,
    confusion_matrix, roc_auc_score, mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression, RidgeCV
from sklearn.svm import SVC, SVR
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier,
    RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor,
    StackingClassifier, StackingRegressor
)
from sklearn.base import clone
import joblib

print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)

# Optional strong models
HAS_XGB = HAS_LGBM = HAS_CAT = HAS_OPTUNA = HAS_SHAP = False
try:
    from xgboost import XGBClassifier, XGBRegressor
    HAS_XGB = True
    import xgboost as xgb
    print('XGBoost:', xgb.__version__)
except Exception as e:
    print('XGBoost unavailable:', e)
try:
    from lightgbm import LGBMClassifier, LGBMRegressor
    HAS_LGBM = True
    import lightgbm as lgb
    print('LightGBM:', lgb.__version__)
except Exception as e:
    print('LightGBM unavailable:', e)
try:
    from catboost import CatBoostClassifier, CatBoostRegressor
    HAS_CAT = True
    print('CatBoost available')
except Exception as e:
    print('CatBoost unavailable:', e)
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    HAS_OPTUNA = True
    print('Optuna:', optuna.__version__)
except Exception as e:
    print('Optuna unavailable:', e)
try:
    import shap
    HAS_SHAP = True
    print('SHAP:', shap.__version__)
except Exception as e:
    print('SHAP unavailable:', e)

# TensorFlow GPU configuration
HAS_TF = False
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    HAS_TF = True
    print('\nTensorFlow:', tf.__version__)
    gpus = tf.config.list_physical_devices('GPU')
    print('TensorFlow GPUs:', gpus)
    if gpus:
        for gpu in gpus:
            try:
                tf.config.experimental.set_memory_growth(gpu, True)
            except Exception:
                pass
        from tensorflow.keras import mixed_precision
        mixed_precision.set_global_policy('mixed_float16')
        print('Mixed precision enabled:', mixed_precision.global_policy())
except Exception as e:
    print('TensorFlow unavailable:', e)

# Save environment versions for reproducibility
versions = {
    'python': sys.version,
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'sklearn': __import__('sklearn').__version__,
    'joblib': joblib.__version__,
    'xgboost_available': HAS_XGB,
    'lightgbm_available': HAS_LGBM,
    'catboost_available': HAS_CAT,
    'optuna_available': HAS_OPTUNA,
    'shap_available': HAS_SHAP,
    'tensorflow_available': HAS_TF,
}
if HAS_TF:
    versions['tensorflow'] = tf.__version__
    versions['tensorflow_gpu_count'] = len(tf.config.list_physical_devices('GPU'))
with open(DIRS['reports'] / 'python_environment_versions.json', 'w', encoding='utf-8') as f:
    json.dump(versions, f, indent=2)
print('\nEnvironment versions saved to:', DIRS['reports'] / 'python_environment_versions.json')


In [ ]:
# ================================================================
# 2. DATASET LOADING
#    Option A: upload CSV manually to Colab files or Google Drive.
#    Option B: download from Kaggle if kaggle.json is configured.
# ================================================================

KAGGLE_DATASET = 'colabsss/electrochemical-heavy-metal-sensor-data'
EXPECTED_CSV_NAME = 'heavy_metal_dataset_with_target.csv'

# Put your CSV path here if stored in Drive, for example:
# CSV_PATH = '/content/drive/MyDrive/heavy_metal_dataset_with_target.csv'
CSV_PATH = None

# If CSV_PATH is None, notebook will search common locations.
possible_paths = []
if CSV_PATH:
    possible_paths.append(Path(CSV_PATH))
possible_paths += [
    Path('/content') / EXPECTED_CSV_NAME,
    Path('/content/drive/MyDrive') / EXPECTED_CSV_NAME if IN_COLAB else Path.cwd() / EXPECTED_CSV_NAME,
    Path.cwd() / EXPECTED_CSV_NAME,
]

csv_file = None
for p in possible_paths:
    if p.exists():
        csv_file = p
        break

if csv_file is None:
    print('CSV not found in common paths.')
    print('Trying Kaggle download. Make sure kaggle.json is uploaded/configured.')
    try:
        !mkdir -p ~/.kaggle
        if Path('/content/kaggle.json').exists():
            !cp /content/kaggle.json ~/.kaggle/kaggle.json
        !chmod 600 ~/.kaggle/kaggle.json
        !kaggle datasets download -d colabsss/electrochemical-heavy-metal-sensor-data -p /content/kaggle_hm --unzip
        files = list(Path('/content/kaggle_hm').glob('*.csv'))
        if not files:
            raise FileNotFoundError('No CSV file found after Kaggle download.')
        csv_file = files[0]
    except Exception as e:
        raise FileNotFoundError(
            'Dataset CSV not found. Upload heavy_metal_dataset_with_target.csv to Colab or Drive, '
            'or configure Kaggle API. Error: ' + str(e)
        )

print('Using CSV:', csv_file)
df_raw = pd.read_csv(csv_file)
df_raw.to_csv(DIRS['raw'] / EXPECTED_CSV_NAME, index=False)
df_raw.to_csv(DIRS['raw'] / 'raw_dataset_backup.csv', index=False)
print('Shape:', df_raw.shape)
display(df_raw.head())
display(df_raw.info())

In [ ]:
# ================================================================
# 3. REGULATORY / LITERATURE THRESHOLDS
# ================================================================

# Main standard used in this notebook.
# WHO_2022 is selected by default because it provides guideline values for the four metals in this dataset.
# EPA_NPDWR can be selected for U.S. regulatory framing; lead/copper are action-level based.
THRESHOLD_STANDARD = 'WHO_2022'  # options: 'WHO_2022', 'EPA_NPDWR'

threshold_records = [
    # WHO drinking-water guideline values, 4th edition incorporating addenda, 2022.
    {'standard': 'WHO_2022', 'metal': 'Cd', 'threshold_mg_L': 0.003, 'basis': 'WHO guideline value for cadmium in drinking water', 'type': 'guideline'},
    {'standard': 'WHO_2022', 'metal': 'Pb', 'threshold_mg_L': 0.010, 'basis': 'WHO provisional guideline value for lead in drinking water', 'type': 'provisional guideline'},
    {'standard': 'WHO_2022', 'metal': 'Hg', 'threshold_mg_L': 0.006, 'basis': 'WHO guideline value for inorganic mercury in drinking water', 'type': 'guideline'},
    {'standard': 'WHO_2022', 'metal': 'Cu', 'threshold_mg_L': 2.000, 'basis': 'WHO health-based/aesthetic guideline value for copper in drinking water', 'type': 'guideline'},

    # EPA National Primary Drinking Water Regulations / Lead and Copper Rule framing.
    {'standard': 'EPA_NPDWR', 'metal': 'Cd', 'threshold_mg_L': 0.005, 'basis': 'EPA MCL for cadmium', 'type': 'MCL'},
    {'standard': 'EPA_NPDWR', 'metal': 'Pb', 'threshold_mg_L': 0.010, 'basis': 'EPA lead action/trigger-level framing; older action level often reported as 0.015 mg/L', 'type': 'action/trigger level'},
    {'standard': 'EPA_NPDWR', 'metal': 'Hg', 'threshold_mg_L': 0.002, 'basis': 'EPA MCL for inorganic mercury', 'type': 'MCL'},
    {'standard': 'EPA_NPDWR', 'metal': 'Cu', 'threshold_mg_L': 1.300, 'basis': 'EPA copper action level', 'type': 'action level'},
]
threshold_df = pd.DataFrame(threshold_records)
threshold_df.to_csv(DIRS['tables'] / 'table_00_regulatory_thresholds.csv', index=False)
threshold_df.to_excel(DIRS['tables'] / 'table_00_regulatory_thresholds.xlsx', index=False)

selected_thresholds = threshold_df[threshold_df['standard'] == THRESHOLD_STANDARD].set_index('metal')['threshold_mg_L'].to_dict()
print('Selected standard:', THRESHOLD_STANDARD)
print(selected_thresholds)
display(threshold_df)

# Citation notes saved for manuscript methods section.
threshold_sources = {
    'WHO_2022': 'WHO Guidelines for drinking-water quality, fourth edition incorporating the first and second addenda, 2022. https://www.who.int/teams/environment-climate-change-and-health/water-sanitation-and-health/water-safety-and-quality/drinking-water-quality-guidelines',
    'EPA_NPDWR': 'U.S. EPA National Primary Drinking Water Regulations. https://www.epa.gov/ground-water-and-drinking-water/national-primary-drinking-water-regulations'
}
with open(DIRS['reports'] / 'threshold_sources.json', 'w', encoding='utf-8') as f:
    json.dump(threshold_sources, f, indent=2)

In [ ]:
# ================================================================
# 4. CLEANING, COLUMN VALIDATION, AND REGULATORY STATUS LABELS
# ================================================================

required_cols = [
    'Metal_Type', 'Concentration_mg_L', 'Frequency_Hz', 'Real_Impedance_ohm',
    'Imag_Impedance_ohm', 'Magnitude_ohm', 'Phase_deg',
    'Charge_Transfer_Resistance_ohm', 'Double_Layer_Capacitance_F',
    'Temperature_C', 'pH', 'Conductivity_uS_cm'
]
missing_required = [c for c in required_cols if c not in df_raw.columns]
if missing_required:
    raise ValueError(f'Missing required columns: {missing_required}. Please adjust column mapping in this notebook.')

df = df_raw.copy()

# Standardize metal strings
df['Metal_Type'] = df['Metal_Type'].astype(str).str.strip()

# Convert all non-metal columns to numeric where possible
for col in df.columns:
    if col != 'Metal_Type':
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Basic cleaning
cleaning_report = {
    'initial_shape': df_raw.shape,
    'missing_before': df.isna().sum().to_dict(),
    'duplicates_before': int(df.duplicated().sum())
}
df = df.drop_duplicates().reset_index(drop=True)

# Impute numeric missing values by median; metal by mode if needed.
num_cols_all = df.select_dtypes(include=[np.number]).columns.tolist()
for col in num_cols_all:
    df[col] = df[col].fillna(df[col].median())
if df['Metal_Type'].isna().any():
    df['Metal_Type'] = df['Metal_Type'].fillna(df['Metal_Type'].mode()[0])

# Regulatory threshold and ratio
if not set(df['Metal_Type'].unique()).issubset(set(selected_thresholds.keys())):
    unknown = sorted(set(df['Metal_Type'].unique()) - set(selected_thresholds.keys()))
    raise ValueError(f'Metal types not covered by selected thresholds: {unknown}')

def get_threshold(metal):
    return float(selected_thresholds[str(metal)])

def contamination_status_from_ratio(ratio):
    # Methodological categories based on exceedance ratio.
    # Safe: <= regulatory/guideline threshold.
    # Low/moderate/heavy: increasing exceedance relative to threshold.
    if ratio <= 1:
        return 'Safe'
    elif ratio <= 5:
        return 'Low contamination'
    elif ratio <= 20:
        return 'Moderate contamination'
    else:
        return 'Heavy contamination'

df['regulatory_threshold_mg_L'] = df['Metal_Type'].map(selected_thresholds).astype(float)
df['exceedance_ratio'] = df['Concentration_mg_L'] / df['regulatory_threshold_mg_L']
df['Contamination_Status_Regulatory'] = df['exceedance_ratio'].apply(contamination_status_from_ratio)
df['Safety_Status_Regulatory'] = np.where(df['exceedance_ratio'] <= 1, 'Safe', 'Unsafe')

cleaning_report['final_shape'] = df.shape
cleaning_report['missing_after'] = df.isna().sum().to_dict()
cleaning_report['duplicates_after'] = int(df.duplicated().sum())
with open(DIRS['reports'] / 'cleaning_report.json', 'w', encoding='utf-8') as f:
    json.dump(cleaning_report, f, indent=2)

# Save cleaned dataset
side_cols = ['regulatory_threshold_mg_L', 'exceedance_ratio', 'Contamination_Status_Regulatory', 'Safety_Status_Regulatory']
df.to_csv(DIRS['processed'] / 'heavy_metal_cleaned_with_regulatory_status.csv', index=False)
df.to_excel(DIRS['processed'] / 'heavy_metal_cleaned_with_regulatory_status.xlsx', index=False)

display(df.head())
print('Status distribution:')
display(df['Contamination_Status_Regulatory'].value_counts().to_frame('count'))
print('Safety distribution:')
display(df['Safety_Status_Regulatory'].value_counts().to_frame('count'))

In [ ]:
# ================================================================
# 5. FEATURE ENGINEERING FOR SENSOR RESPONSE MODELING
# ================================================================

EPS = 1e-9
fe = df.copy()

# Physics-inspired electrochemical/impedance features
fe['Real_Impedance_abs'] = fe['Real_Impedance_ohm'].abs()
fe['Imag_Impedance_abs'] = fe['Imag_Impedance_ohm'].abs()
fe['Impedance_real_imag_ratio'] = fe['Real_Impedance_abs'] / (fe['Imag_Impedance_abs'] + EPS)
fe['Impedance_magnitude_log1p'] = np.log1p(fe['Magnitude_ohm'].clip(lower=0))
fe['Frequency_log10'] = np.log10(fe['Frequency_Hz'].clip(lower=EPS))
fe['Charge_Transfer_Resistance_log1p'] = np.log1p(fe['Charge_Transfer_Resistance_ohm'].clip(lower=0))
fe['Capacitance_log10_abs'] = np.log10(fe['Double_Layer_Capacitance_F'].abs() + EPS)
fe['Conductance_proxy'] = 1.0 / (fe['Magnitude_ohm'].abs() + EPS)
fe['Phase_rad'] = np.deg2rad(fe['Phase_deg'])
fe['Phase_sin'] = np.sin(fe['Phase_rad'])
fe['Phase_cos'] = np.cos(fe['Phase_rad'])
fe['Temp_pH_interaction'] = fe['Temperature_C'] * fe['pH']
fe['Conductivity_pH_ratio'] = fe['Conductivity_uS_cm'] / (fe['pH'].abs() + EPS)
fe['Rct_Cdl_product'] = fe['Charge_Transfer_Resistance_ohm'] * fe['Double_Layer_Capacitance_F']
fe['Normalized_Rct_by_Z'] = fe['Charge_Transfer_Resistance_ohm'] / (fe['Magnitude_ohm'].abs() + EPS)

# Do not include target leakage columns as model inputs.
TARGET_COLS = [
    'Metal_Type', 'Target', 'Concentration_mg_L',
    'regulatory_threshold_mg_L', 'exceedance_ratio',
    'Contamination_Status_Regulatory', 'Safety_Status_Regulatory'
]
feature_cols = [c for c in fe.columns if c not in TARGET_COLS]
feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(fe[c])]

fe.to_csv(DIRS['processed'] / 'heavy_metal_feature_engineered_regulatory.csv', index=False)
fe.to_excel(DIRS['processed'] / 'heavy_metal_feature_engineered_regulatory.xlsx', index=False)

schema = {
    'feature_columns': feature_cols,
    'target_columns': TARGET_COLS,
    'metal_column': 'Metal_Type',
    'concentration_column': 'Concentration_mg_L',
    'status_column': 'Contamination_Status_Regulatory',
    'safety_column': 'Safety_Status_Regulatory',
    'threshold_standard': THRESHOLD_STANDARD,
    'selected_thresholds_mg_L': selected_thresholds,
    'status_rule': 'Safe if concentration/threshold <= 1; Low if <=5; Moderate if <=20; Heavy if >20.'
}
with open(DIRS['models'] / 'input_schema.json', 'w', encoding='utf-8') as f:
    json.dump(schema, f, indent=2)

print('Number of features:', len(feature_cols))
print(feature_cols)
display(fe[feature_cols + ['Metal_Type', 'Concentration_mg_L', 'Contamination_Status_Regulatory']].head())

In [ ]:
# ================================================================
# 6. EXPLORATORY DATA ANALYSIS AND PAPER FIGURES
# ================================================================

# Helper functions

def save_fig(name, dpi=300):
    path = DIRS['figures'] / name
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches='tight')
    plt.show()
    return path

# Summary statistics tables
summary_stats = fe[feature_cols + ['Concentration_mg_L', 'exceedance_ratio']].describe().T
summary_stats.to_csv(DIRS['tables'] / 'table_01_summary_statistics.csv')
summary_stats.to_excel(DIRS['tables'] / 'table_01_summary_statistics.xlsx')

missing_table = pd.DataFrame({'missing_count': fe.isna().sum(), 'missing_percent': fe.isna().mean()*100})
missing_table.to_csv(DIRS['tables'] / 'table_02_missing_values.csv')
missing_table.to_excel(DIRS['tables'] / 'table_02_missing_values.xlsx')

metal_status_table = pd.crosstab(fe['Metal_Type'], fe['Contamination_Status_Regulatory'])
metal_status_table.to_csv(DIRS['tables'] / 'table_03_metal_vs_regulatory_status.csv')
metal_status_table.to_excel(DIRS['tables'] / 'table_03_metal_vs_regulatory_status.xlsx')

# Figure 1: digital twin framework
plt.figure(figsize=(12, 7))
steps = [
    'Kaggle Electrochemical\nHeavy Metal Sensor Data',
    'Cleaning + Feature\nEngineering',
    'Sensor Response\nRepresentation',
    'Metal Classifier\nML/DL',
    'Concentration\nRegressor',
    'Regulatory Threshold\nEngine',
    'Digital Twin\nPrediction',
    'Next.js Dashboard\n+ Render API'
]
xs = np.linspace(0.05, 0.95, len(steps))
y = 0.55
for i, (x, s) in enumerate(zip(xs, steps)):
    plt.gca().add_patch(plt.Rectangle((x-0.055, y-0.11), 0.11, 0.22, fill=False, linewidth=1.5))
    plt.text(x, y, s, ha='center', va='center', fontsize=8)
    if i < len(steps)-1:
        plt.arrow(x+0.06, y, xs[i+1]-x-0.12, 0, length_includes_head=True, head_width=0.025, head_length=0.015)
plt.axis('off')
plt.title('Digital Twin Framework with Regulatory Threshold-Based Status Engine')
save_fig('figure_01_revised_digital_twin_framework.png')

# Distribution figures
plt.figure(figsize=(8, 5))
fe['Metal_Type'].value_counts().sort_index().plot(kind='bar')
plt.title('Metal Type Distribution')
plt.xlabel('Metal type')
plt.ylabel('Count')
save_fig('figure_02_metal_distribution.png')

plt.figure(figsize=(8, 5))
for metal, group in fe.groupby('Metal_Type'):
    plt.hist(group['Concentration_mg_L'], alpha=0.45, bins=30, label=metal)
plt.title('Concentration Distribution by Metal')
plt.xlabel('Concentration (mg/L)')
plt.ylabel('Frequency')
plt.legend()
save_fig('figure_03_concentration_distribution_by_metal.png')

plt.figure(figsize=(8, 5))
fe['Contamination_Status_Regulatory'].value_counts().plot(kind='bar')
plt.title(f'Regulatory Contamination Status Distribution ({THRESHOLD_STANDARD})')
plt.xlabel('Status')
plt.ylabel('Count')
plt.xticks(rotation=30, ha='right')
save_fig('figure_04_regulatory_status_distribution.png')

plt.figure(figsize=(9, 7))
corr_cols = feature_cols[:]
if len(corr_cols) > 18:
    # Keep most correlated with concentration for readable matrix
    corrs = fe[corr_cols + ['Concentration_mg_L']].corr(numeric_only=True)['Concentration_mg_L'].abs().sort_values(ascending=False)
    corr_cols = [c for c in corrs.index if c != 'Concentration_mg_L'][:18]
corr = fe[corr_cols + ['Concentration_mg_L']].corr(numeric_only=True)
plt.imshow(corr, aspect='auto')
plt.colorbar(label='Pearson correlation')
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90, fontsize=7)
plt.yticks(range(len(corr.index)), corr.index, fontsize=7)
plt.title('Correlation Matrix of Sensor Features and Concentration')
save_fig('figure_05_correlation_matrix.png')

plt.figure(figsize=(8, 5))
for metal, group in fe.groupby('Metal_Type'):
    plt.scatter(group['Magnitude_ohm'], group['Concentration_mg_L'], alpha=0.5, s=15, label=metal)
plt.xlabel('Magnitude (ohm)')
plt.ylabel('Concentration (mg/L)')
plt.title('Sensor Magnitude vs Heavy Metal Concentration')
plt.legend()
save_fig('figure_06_magnitude_vs_concentration.png')

In [ ]:
# ================================================================
# 7. TRAIN/TEST SPLIT AND ENCODERS
# ================================================================

X = fe[feature_cols].copy()
y_metal = fe['Metal_Type'].copy()
y_conc = fe['Concentration_mg_L'].copy()
y_status = fe['Contamination_Status_Regulatory'].copy()
y_safety = fe['Safety_Status_Regulatory'].copy()

metal_le = LabelEncoder()
status_le = LabelEncoder()
safety_le = LabelEncoder()
y_metal_enc = metal_le.fit_transform(y_metal)
y_status_enc = status_le.fit_transform(y_status)
y_safety_enc = safety_le.fit_transform(y_safety)

# Stratify by metal + safety/status when possible to keep split balanced.
stratify_key = y_metal.astype(str) + '_' + y_status.astype(str)
min_class = stratify_key.value_counts().min()
if min_class < 2:
    stratify_key = y_metal

X_train, X_test, ym_train, ym_test, yc_train, yc_test, ys_train, ys_test, ysafe_train, ysafe_test, idx_train, idx_test = train_test_split(
    X, y_metal_enc, y_conc, y_status_enc, y_safety_enc, fe.index,
    test_size=0.20,
    random_state=SEED,
    stratify=stratify_key
)

# Base preprocessing pipeline for numeric features
numeric_preprocessor = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

joblib.dump(metal_le, DIRS['models'] / 'metal_label_encoder.joblib')
joblib.dump(status_le, DIRS['models'] / 'contamination_status_label_encoder.joblib')
joblib.dump(safety_le, DIRS['models'] / 'safety_status_label_encoder.joblib')

split_info = {
    'n_train': len(X_train),
    'n_test': len(X_test),
    'metal_classes': metal_le.classes_.tolist(),
    'status_classes': status_le.classes_.tolist(),
    'safety_classes': safety_le.classes_.tolist(),
    'stratify_min_class': int(min_class),
}
with open(DIRS['reports'] / 'split_info.json', 'w', encoding='utf-8') as f:
    json.dump(split_info, f, indent=2)

print(split_info)

In [ ]:
# ================================================================
# 8. MODEL ZOO: STRONG TABULAR BASELINES
# ================================================================

def make_classification_models(num_classes):
    models = {
        'LogisticRegression': LogisticRegression(max_iter=3000, class_weight='balanced', random_state=SEED),
        'RandomForest': RandomForestClassifier(n_estimators=600, class_weight='balanced', random_state=SEED, n_jobs=-1),
        'ExtraTrees': ExtraTreesClassifier(n_estimators=800, class_weight='balanced', random_state=SEED, n_jobs=-1),
        'GradientBoosting': GradientBoostingClassifier(random_state=SEED),
        'HistGradientBoosting': HistGradientBoostingClassifier(random_state=SEED, max_iter=400),
        'SVC_RBF': SVC(C=10, gamma='scale', kernel='rbf', probability=True, class_weight='balanced', random_state=SEED),
    }
    if HAS_XGB:
        models['XGBoost'] = XGBClassifier(
            n_estimators=600, max_depth=5, learning_rate=0.035, subsample=0.9, colsample_bytree=0.9,
            objective='multi:softprob' if num_classes > 2 else 'binary:logistic',
            eval_metric='mlogloss' if num_classes > 2 else 'logloss', random_state=SEED, n_jobs=-1,
            tree_method='hist', device='cuda' if gpu_info['nvidia_smi_available'] else 'cpu'
        )
    if HAS_LGBM:
        models['LightGBM'] = LGBMClassifier(n_estimators=700, learning_rate=0.035, num_leaves=31, random_state=SEED, n_jobs=-1, verbose=-1)
    if HAS_CAT:
        models['CatBoost'] = CatBoostClassifier(iterations=700, learning_rate=0.035, depth=6, loss_function='MultiClass' if num_classes > 2 else 'Logloss', random_seed=SEED, verbose=False)
    return models


def make_regression_models():
    models = {
        'RandomForest': RandomForestRegressor(n_estimators=700, random_state=SEED, n_jobs=-1),
        'ExtraTrees': ExtraTreesRegressor(n_estimators=900, random_state=SEED, n_jobs=-1),
        'GradientBoosting': GradientBoostingRegressor(random_state=SEED, n_estimators=700, learning_rate=0.035, max_depth=3),
        'HistGradientBoosting': HistGradientBoostingRegressor(random_state=SEED, max_iter=600, learning_rate=0.035),
        'SVR_RBF': SVR(C=50, epsilon=0.02, gamma='scale'),
    }
    if HAS_XGB:
        models['XGBoost'] = XGBRegressor(
            n_estimators=800, max_depth=5, learning_rate=0.025, subsample=0.9, colsample_bytree=0.9,
            objective='reg:squarederror', random_state=SEED, n_jobs=-1,
            tree_method='hist', device='cuda' if gpu_info['nvidia_smi_available'] else 'cpu'
        )
    if HAS_LGBM:
        models['LightGBM'] = LGBMRegressor(n_estimators=900, learning_rate=0.025, num_leaves=31, random_state=SEED, n_jobs=-1, verbose=-1)
    if HAS_CAT:
        models['CatBoost'] = CatBoostRegressor(iterations=900, learning_rate=0.025, depth=6, loss_function='RMSE', random_seed=SEED, verbose=False)
    return models

classification_models = make_classification_models(num_classes=len(metal_le.classes_))
status_models = make_classification_models(num_classes=len(status_le.classes_))
regression_models = make_regression_models()

print('Classification models:', list(classification_models.keys()))
print('Regression models:', list(regression_models.keys()))

In [ ]:
# ================================================================
# 9. REPEATED CROSS-VALIDATION FOR PAPER-QUALITY EVALUATION
# ================================================================

RUN_REPEATED_CV = True
CV_SPLITS = 5
CV_REPEATS = 3

cv_tables = {}

if RUN_REPEATED_CV:
    # Metal classification CV
    metal_cv = RepeatedStratifiedKFold(n_splits=CV_SPLITS, n_repeats=CV_REPEATS, random_state=SEED)
    records = []
    for name, model in classification_models.items():
        print('CV metal:', name)
        pipe = Pipeline([('prep', clone(numeric_preprocessor)), ('model', clone(model))])
        scores = cross_validate(pipe, X, y_metal_enc, cv=metal_cv, scoring=['accuracy', 'f1_macro'], n_jobs=-1, error_score='raise')
        records.append({
            'model': name,
            'accuracy_mean': scores['test_accuracy'].mean(),
            'accuracy_std': scores['test_accuracy'].std(),
            'macro_f1_mean': scores['test_f1_macro'].mean(),
            'macro_f1_std': scores['test_f1_macro'].std(),
        })
    metal_cv_df = pd.DataFrame(records).sort_values('macro_f1_mean', ascending=False)
    cv_tables['metal_classification_cv'] = metal_cv_df
    metal_cv_df.to_csv(DIRS['tables'] / 'table_04_repeated_cv_metal_classification.csv', index=False)
    metal_cv_df.to_excel(DIRS['tables'] / 'table_04_repeated_cv_metal_classification.xlsx', index=False)
    display(metal_cv_df)

    # Direct status classification CV as a baseline only.
    status_cv = RepeatedStratifiedKFold(n_splits=CV_SPLITS, n_repeats=CV_REPEATS, random_state=SEED)
    records = []
    for name, model in status_models.items():
        print('CV direct status baseline:', name)
        pipe = Pipeline([('prep', clone(numeric_preprocessor)), ('model', clone(model))])
        scores = cross_validate(pipe, X, y_status_enc, cv=status_cv, scoring=['accuracy', 'f1_macro'], n_jobs=-1, error_score='raise')
        records.append({
            'model': name,
            'accuracy_mean': scores['test_accuracy'].mean(),
            'accuracy_std': scores['test_accuracy'].std(),
            'macro_f1_mean': scores['test_f1_macro'].mean(),
            'macro_f1_std': scores['test_f1_macro'].std(),
            'note': 'Direct status classifier baseline; primary status pipeline is concentration-regression-based.'
        })
    status_cv_df = pd.DataFrame(records).sort_values('macro_f1_mean', ascending=False)
    cv_tables['direct_status_classification_cv'] = status_cv_df
    status_cv_df.to_csv(DIRS['tables'] / 'table_05_repeated_cv_direct_status_baseline.csv', index=False)
    status_cv_df.to_excel(DIRS['tables'] / 'table_05_repeated_cv_direct_status_baseline.xlsx', index=False)
    display(status_cv_df)

    # Concentration regression CV
    reg_cv = RepeatedKFold(n_splits=CV_SPLITS, n_repeats=CV_REPEATS, random_state=SEED)
    records = []
    for name, model in regression_models.items():
        print('CV regression:', name)
        pipe = Pipeline([('prep', clone(numeric_preprocessor)), ('model', clone(model))])
        scores = cross_validate(pipe, X, y_conc, cv=reg_cv, scoring=['r2', 'neg_mean_absolute_error', 'neg_root_mean_squared_error'], n_jobs=-1, error_score='raise')
        records.append({
            'model': name,
            'r2_mean': scores['test_r2'].mean(),
            'r2_std': scores['test_r2'].std(),
            'mae_mean': -scores['test_neg_mean_absolute_error'].mean(),
            'mae_std': scores['test_neg_mean_absolute_error'].std(),
            'rmse_mean': -scores['test_neg_root_mean_squared_error'].mean(),
            'rmse_std': scores['test_neg_root_mean_squared_error'].std(),
        })
    reg_cv_df = pd.DataFrame(records).sort_values('r2_mean', ascending=False)
    cv_tables['concentration_regression_cv'] = reg_cv_df
    reg_cv_df.to_csv(DIRS['tables'] / 'table_06_repeated_cv_concentration_regression.csv', index=False)
    reg_cv_df.to_excel(DIRS['tables'] / 'table_06_repeated_cv_concentration_regression.xlsx', index=False)
    display(reg_cv_df)

In [ ]:
# ================================================================
# 10. OPTIONAL OPTUNA HYPERPARAMETER TUNING FOR TOP MODELS
# ================================================================

RUN_OPTUNA = True
N_OPTUNA_TRIALS = 50

best_tuned_models = {}

if RUN_OPTUNA and HAS_OPTUNA:
    # Tune ExtraTrees classifier for metal classification because it is robust for tabular data.
    def objective_metal_etc(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 400, 1200),
            'max_depth': trial.suggest_int('max_depth', 4, 40),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 12),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 8),
            'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
            'class_weight': 'balanced',
            'random_state': SEED,
            'n_jobs': -1
        }
        model = ExtraTreesClassifier(**params)
        pipe = Pipeline([('prep', clone(numeric_preprocessor)), ('model', model)])
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
        scores = cross_validate(pipe, X_train, ym_train, cv=cv, scoring='f1_macro', n_jobs=-1, error_score='raise')
        return scores['test_score'].mean()

    print('Tuning ExtraTrees metal classifier...')
    study_metal = optuna.create_study(direction='maximize')
    study_metal.optimize(objective_metal_etc, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    best_metal_model = ExtraTreesClassifier(**study_metal.best_params, class_weight='balanced', random_state=SEED, n_jobs=-1)
    best_tuned_models['Optuna_ExtraTrees_Metal'] = best_metal_model

    # Tune ExtraTrees regressor for concentration estimation.
    def objective_reg_etr(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 500, 1400),
            'max_depth': trial.suggest_int('max_depth', 4, 50),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 12),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 8),
            'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
            'random_state': SEED,
            'n_jobs': -1
        }
        model = ExtraTreesRegressor(**params)
        pipe = Pipeline([('prep', clone(numeric_preprocessor)), ('model', model)])
        cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
        scores = cross_validate(pipe, X_train, yc_train, cv=cv, scoring='r2', n_jobs=-1, error_score='raise')
        return scores['test_score'].mean()

    print('Tuning ExtraTrees concentration regressor...')
    study_reg = optuna.create_study(direction='maximize')
    study_reg.optimize(objective_reg_etr, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    best_reg_model = ExtraTreesRegressor(**study_reg.best_params, random_state=SEED, n_jobs=-1)
    best_tuned_models['Optuna_ExtraTrees_Regressor'] = best_reg_model

    optuna_summary = {
        'metal_best_params': study_metal.best_params,
        'metal_best_f1_macro_cv': study_metal.best_value,
        'regression_best_params': study_reg.best_params,
        'regression_best_r2_cv': study_reg.best_value,
    }
    with open(DIRS['reports'] / 'optuna_summary.json', 'w', encoding='utf-8') as f:
        json.dump(optuna_summary, f, indent=2)
    print(optuna_summary)
else:
    print('Optuna skipped or unavailable.')

In [ ]:
# ================================================================
# 11. FIT FINAL ML MODELS, STACKING ENSEMBLES, AND TEST EVALUATION
# ================================================================

# Add tuned models to model dictionaries
if 'Optuna_ExtraTrees_Metal' in best_tuned_models:
    classification_models['Optuna_ExtraTrees'] = best_tuned_models['Optuna_ExtraTrees_Metal']
if 'Optuna_ExtraTrees_Regressor' in best_tuned_models:
    regression_models['Optuna_ExtraTrees'] = best_tuned_models['Optuna_ExtraTrees_Regressor']

# Stacking ensembles from strong base models
# Use models that are robust and reasonably fast.
stack_clf_estimators = []
for key in ['ExtraTrees', 'RandomForest', 'HistGradientBoosting', 'XGBoost', 'LightGBM', 'CatBoost']:
    if key in classification_models:
        stack_clf_estimators.append((key.lower(), clone(classification_models[key])))
if len(stack_clf_estimators) >= 2:
    classification_models['StackingEnsemble'] = StackingClassifier(
        estimators=stack_clf_estimators[:4],
        final_estimator=LogisticRegression(max_iter=2000, class_weight='balanced'),
        stack_method='auto', n_jobs=-1, passthrough=False
    )

stack_reg_estimators = []
for key in ['ExtraTrees', 'RandomForest', 'HistGradientBoosting', 'XGBoost', 'LightGBM', 'CatBoost']:
    if key in regression_models:
        stack_reg_estimators.append((key.lower(), clone(regression_models[key])))
if len(stack_reg_estimators) >= 2:
    regression_models['StackingEnsemble'] = StackingRegressor(
        estimators=stack_reg_estimators[:4],
        final_estimator=RidgeCV(),
        n_jobs=-1, passthrough=False
    )

# Metal classification final evaluation
metal_results = []
fitted_metal_models = {}
for name, model in classification_models.items():
    print('Fitting metal model:', name)
    pipe = Pipeline([('prep', clone(numeric_preprocessor)), ('model', clone(model))])
    pipe.fit(X_train, ym_train)
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test) if hasattr(pipe, 'predict_proba') else None
    acc = accuracy_score(ym_test, pred)
    f1 = f1_score(ym_test, pred, average='macro')
    auc = np.nan
    if proba is not None and len(np.unique(ym_test)) > 2:
        try:
            auc = roc_auc_score(ym_test, proba, multi_class='ovr', average='macro')
        except Exception:
            pass
    metal_results.append({'model': name, 'accuracy': acc, 'macro_f1': f1, 'macro_roc_auc_ovr': auc})
    fitted_metal_models[name] = pipe
    joblib.dump(pipe, DIRS['models'] / f'metal_classifier_{name}.joblib')

metal_results_df = pd.DataFrame(metal_results).sort_values('macro_f1', ascending=False)
metal_results_df.to_csv(DIRS['tables'] / 'table_07_test_metal_classification.csv', index=False)
metal_results_df.to_excel(DIRS['tables'] / 'table_07_test_metal_classification.xlsx', index=False)
display(metal_results_df)

# Direct status classification baseline final evaluation
status_results = []
fitted_status_models = {}
for name, model in status_models.items():
    print('Fitting direct status baseline:', name)
    pipe = Pipeline([('prep', clone(numeric_preprocessor)), ('model', clone(model))])
    pipe.fit(X_train, ys_train)
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test) if hasattr(pipe, 'predict_proba') else None
    acc = accuracy_score(ys_test, pred)
    f1 = f1_score(ys_test, pred, average='macro')
    auc = np.nan
    if proba is not None and len(np.unique(ys_test)) > 2:
        try:
            auc = roc_auc_score(ys_test, proba, multi_class='ovr', average='macro')
        except Exception:
            pass
    status_results.append({'model': name, 'accuracy': acc, 'macro_f1': f1, 'macro_roc_auc_ovr': auc, 'role': 'direct status baseline'})
    fitted_status_models[name] = pipe
    joblib.dump(pipe, DIRS['models'] / f'direct_status_classifier_{name}.joblib')

status_results_df = pd.DataFrame(status_results).sort_values('macro_f1', ascending=False)
status_results_df.to_csv(DIRS['tables'] / 'table_08_test_direct_status_baseline.csv', index=False)
status_results_df.to_excel(DIRS['tables'] / 'table_08_test_direct_status_baseline.xlsx', index=False)
display(status_results_df)

# Concentration regression final evaluation
reg_results = []
fitted_reg_models = {}
for name, model in regression_models.items():
    print('Fitting concentration regressor:', name)
    pipe = Pipeline([('prep', clone(numeric_preprocessor)), ('model', clone(model))])
    pipe.fit(X_train, yc_train)
    pred = pipe.predict(X_test)
    pred = np.clip(pred, 0, None)
    mae = mean_absolute_error(yc_test, pred)
    rmse = np.sqrt(mean_squared_error(yc_test, pred))
    r2 = r2_score(yc_test, pred)
    reg_results.append({'model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2})
    fitted_reg_models[name] = pipe
    joblib.dump(pipe, DIRS['models'] / f'concentration_regressor_{name}.joblib')

reg_results_df = pd.DataFrame(reg_results).sort_values('R2', ascending=False)
reg_results_df.to_csv(DIRS['tables'] / 'table_09_test_concentration_regression.csv', index=False)
reg_results_df.to_excel(DIRS['tables'] / 'table_09_test_concentration_regression.xlsx', index=False)
display(reg_results_df)

best_metal_name = metal_results_df.iloc[0]['model']
best_reg_name = reg_results_df.iloc[0]['model']
best_status_direct_name = status_results_df.iloc[0]['model']
print('Best metal model:', best_metal_name)
print('Best concentration regressor:', best_reg_name)
print('Best direct status baseline:', best_status_direct_name)

best_metal_model = fitted_metal_models[best_metal_name]
best_reg_model = fitted_reg_models[best_reg_name]
best_direct_status_model = fitted_status_models[best_status_direct_name]

joblib.dump(best_metal_model, DIRS['models'] / 'best_metal_classifier.joblib')
joblib.dump(best_reg_model, DIRS['models'] / 'best_concentration_regressor.joblib')
joblib.dump(best_direct_status_model, DIRS['models'] / 'best_direct_status_classifier_baseline.joblib')

In [ ]:
# ================================================================
# 12. PRIMARY DIGITAL TWIN PIPELINE:
#     SENSOR FEATURES -> METAL PREDICTION + CONCENTRATION PREDICTION -> REGULATORY STATUS
# ================================================================

def status_from_metal_concentration(metal, concentration, threshold_dict=selected_thresholds):
    metal = str(metal)
    threshold = float(threshold_dict[metal])
    conc = max(float(concentration), 0.0)
    ratio = conc / threshold
    status = contamination_status_from_ratio(ratio)
    safety = 'Safe' if ratio <= 1 else 'Unsafe'
    return status, safety, ratio, threshold

# Test-set predictions
pred_metal_enc = best_metal_model.predict(X_test)
pred_metal_label = metal_le.inverse_transform(pred_metal_enc)
true_metal_label = metal_le.inverse_transform(ym_test)

pred_concentration = np.clip(best_reg_model.predict(X_test), 0, None)
true_concentration = np.array(yc_test)

# Primary: predicted metal + predicted concentration
pred_status_label = []
pred_safety_label = []
pred_ratio = []
pred_threshold = []
for m, c in zip(pred_metal_label, pred_concentration):
    s, safe, ratio, thr = status_from_metal_concentration(m, c)
    pred_status_label.append(s)
    pred_safety_label.append(safe)
    pred_ratio.append(ratio)
    pred_threshold.append(thr)

# Oracle-metal variant: true metal + predicted concentration, useful to separate metal classification error from regression error.
oracle_status_label = []
oracle_safety_label = []
for m, c in zip(true_metal_label, pred_concentration):
    s, safe, _, _ = status_from_metal_concentration(m, c)
    oracle_status_label.append(s)
    oracle_safety_label.append(safe)

true_status_label = status_le.inverse_transform(ys_test)
true_safety_label = safety_le.inverse_transform(ysafe_test)

# Metrics for primary pipeline
pipeline_status_acc = accuracy_score(true_status_label, pred_status_label)
pipeline_status_f1 = f1_score(true_status_label, pred_status_label, average='macro')
pipeline_safety_acc = accuracy_score(true_safety_label, pred_safety_label)
pipeline_safety_f1 = f1_score(true_safety_label, pred_safety_label, average='macro')

oracle_status_acc = accuracy_score(true_status_label, oracle_status_label)
oracle_status_f1 = f1_score(true_status_label, oracle_status_label, average='macro')
oracle_safety_acc = accuracy_score(true_safety_label, oracle_safety_label)
oracle_safety_f1 = f1_score(true_safety_label, oracle_safety_label, average='macro')

pipeline_metrics = pd.DataFrame([
    {'pipeline': 'Predicted metal + predicted concentration', 'status_accuracy': pipeline_status_acc, 'status_macro_f1': pipeline_status_f1, 'safety_accuracy': pipeline_safety_acc, 'safety_macro_f1': pipeline_safety_f1},
    {'pipeline': 'True/oracle metal + predicted concentration', 'status_accuracy': oracle_status_acc, 'status_macro_f1': oracle_status_f1, 'safety_accuracy': oracle_safety_acc, 'safety_macro_f1': oracle_safety_f1},
    {'pipeline': 'Direct status classifier baseline', 'status_accuracy': status_results_df.iloc[0]['accuracy'], 'status_macro_f1': status_results_df.iloc[0]['macro_f1'], 'safety_accuracy': np.nan, 'safety_macro_f1': np.nan},
])
pipeline_metrics.to_csv(DIRS['tables'] / 'table_10_digital_twin_status_pipeline_metrics.csv', index=False)
pipeline_metrics.to_excel(DIRS['tables'] / 'table_10_digital_twin_status_pipeline_metrics.xlsx', index=False)
display(pipeline_metrics)

# Detailed prediction table for paper and dashboard
pred_table = X_test.copy()
pred_table['true_metal'] = true_metal_label
pred_table['predicted_metal'] = pred_metal_label
pred_table['true_concentration_mg_L'] = true_concentration
pred_table['predicted_concentration_mg_L'] = pred_concentration
pred_table['true_status'] = true_status_label
pred_table['predicted_status_pipeline'] = pred_status_label
pred_table['true_safety'] = true_safety_label
pred_table['predicted_safety_pipeline'] = pred_safety_label
pred_table['predicted_exceedance_ratio'] = pred_ratio
pred_table['predicted_threshold_mg_L'] = pred_threshold
pred_table.to_csv(DIRS['processed'] / 'test_predictions_digital_twin_pipeline.csv', index=False)
pred_table.head(30).to_csv(DIRS['tables'] / 'table_11_sample_digital_twin_predictions.csv', index=False)
pred_table.head(30).to_excel(DIRS['tables'] / 'table_11_sample_digital_twin_predictions.xlsx', index=False)
display(pred_table.head())

# Full classification reports
with open(DIRS['reports'] / 'classification_report_metal.txt', 'w', encoding='utf-8') as f:
    f.write(classification_report(ym_test, pred_metal_enc, target_names=metal_le.classes_))
with open(DIRS['reports'] / 'classification_report_pipeline_status.txt', 'w', encoding='utf-8') as f:
    f.write(classification_report(true_status_label, pred_status_label))
with open(DIRS['reports'] / 'classification_report_pipeline_safety.txt', 'w', encoding='utf-8') as f:
    f.write(classification_report(true_safety_label, pred_safety_label))

In [ ]:
# ================================================================
# 13. DEEP LEARNING MODELS WITH GPU/CUDA SUPPORT
#     MLP and 1D-CNN are included to match the abstract, but tabular ensemble may remain stronger.
# ================================================================

RUN_DEEP_LEARNING = True and HAS_TF
EPOCHS = 350
BATCH_SIZE = 64 if gpu_info['nvidia_smi_available'] else 32

# Prepare scaled arrays for DL
scaler_dl = StandardScaler()
X_train_scaled = scaler_dl.fit_transform(X_train)
X_test_scaled = scaler_dl.transform(X_test)
joblib.dump(scaler_dl, DIRS['models'] / 'deep_learning_scaler.joblib')

# Train-validation split for DL
X_tr, X_val, ym_tr, ym_val, yc_tr, yc_val, ys_tr, ys_val = train_test_split(
    X_train_scaled, ym_train, yc_train, ys_train,
    test_size=0.20, random_state=SEED, stratify=ym_train
)

if RUN_DEEP_LEARNING:
    tf.keras.utils.set_random_seed(SEED)

    def build_mlp_classifier(input_dim, num_classes):
        inputs = keras.Input(shape=(input_dim,))
        x = layers.Dense(256, activation='relu')(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.25)(x)
        x = layers.Dense(128, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.20)(x)
        x = layers.Dense(64, activation='relu')(x)
        outputs = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)
        model = keras.Model(inputs, outputs)
        model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        return model

    def build_mlp_regressor(input_dim):
        inputs = keras.Input(shape=(input_dim,))
        x = layers.Dense(256, activation='relu')(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.20)(x)
        x = layers.Dense(128, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.15)(x)
        x = layers.Dense(64, activation='relu')(x)
        outputs = layers.Dense(1, dtype='float32')(x)
        model = keras.Model(inputs, outputs)
        model.compile(optimizer=keras.optimizers.Adam(learning_rate=8e-4), loss='mse', metrics=[keras.metrics.MeanAbsoluteError(name='mae')])
        return model

    def build_cnn1d_classifier(input_dim, num_classes):
        inputs = keras.Input(shape=(input_dim, 1))
        x = layers.Conv1D(64, 3, padding='same', activation='relu')(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.Conv1D(128, 3, padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.GlobalAveragePooling1D()(x)
        x = layers.Dense(96, activation='relu')(x)
        x = layers.Dropout(0.20)(x)
        outputs = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)
        model = keras.Model(inputs, outputs)
        model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        return model

    callbacks = [
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=35, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=15, min_lr=1e-6)
    ]

    dl_results = []
    device_name = '/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'
    print('Training DL models on:', device_name)
    with tf.device(device_name):
        # MLP metal classifier
        mlp_metal = build_mlp_classifier(X_train_scaled.shape[1], len(metal_le.classes_))
        hist_mlp_metal = mlp_metal.fit(X_tr, ym_tr, validation_data=(X_val, ym_val), epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=callbacks, verbose=0)
        pred = np.argmax(mlp_metal.predict(X_test_scaled, verbose=0), axis=1)
        dl_results.append({'model': 'MLP_metal_classifier', 'task': 'metal_classification', 'accuracy': accuracy_score(ym_test, pred), 'macro_f1': f1_score(ym_test, pred, average='macro'), 'R2': np.nan, 'MAE': np.nan, 'RMSE': np.nan})
        mlp_metal.save(DIRS['models'] / 'mlp_metal_classifier.keras')

        # MLP direct status classifier baseline
        mlp_status = build_mlp_classifier(X_train_scaled.shape[1], len(status_le.classes_))
        hist_mlp_status = mlp_status.fit(X_tr, ys_tr, validation_data=(X_val, ys_val), epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=callbacks, verbose=0)
        pred = np.argmax(mlp_status.predict(X_test_scaled, verbose=0), axis=1)
        dl_results.append({'model': 'MLP_direct_status_classifier', 'task': 'direct_status_baseline', 'accuracy': accuracy_score(ys_test, pred), 'macro_f1': f1_score(ys_test, pred, average='macro'), 'R2': np.nan, 'MAE': np.nan, 'RMSE': np.nan})
        mlp_status.save(DIRS['models'] / 'mlp_direct_status_classifier.keras')

        # MLP concentration regressor
        mlp_reg = build_mlp_regressor(X_train_scaled.shape[1])
        hist_mlp_reg = mlp_reg.fit(X_tr, yc_tr, validation_data=(X_val, yc_val), epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=callbacks, verbose=0)
        pred = np.clip(mlp_reg.predict(X_test_scaled, verbose=0).ravel(), 0, None)
        dl_results.append({'model': 'MLP_concentration_regressor', 'task': 'concentration_regression', 'accuracy': np.nan, 'macro_f1': np.nan, 'R2': r2_score(yc_test, pred), 'MAE': mean_absolute_error(yc_test, pred), 'RMSE': np.sqrt(mean_squared_error(yc_test, pred))})
        mlp_reg.save(DIRS['models'] / 'mlp_concentration_regressor.keras')

        # 1D-CNN metal classifier
        X_tr_cnn = X_tr[..., None]
        X_val_cnn = X_val[..., None]
        X_test_cnn = X_test_scaled[..., None]
        cnn_metal = build_cnn1d_classifier(X_train_scaled.shape[1], len(metal_le.classes_))
        hist_cnn_metal = cnn_metal.fit(X_tr_cnn, ym_tr, validation_data=(X_val_cnn, ym_val), epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=callbacks, verbose=0)
        pred = np.argmax(cnn_metal.predict(X_test_cnn, verbose=0), axis=1)
        dl_results.append({'model': 'CNN1D_metal_classifier', 'task': 'metal_classification', 'accuracy': accuracy_score(ym_test, pred), 'macro_f1': f1_score(ym_test, pred, average='macro'), 'R2': np.nan, 'MAE': np.nan, 'RMSE': np.nan})
        cnn_metal.save(DIRS['models'] / 'cnn1d_metal_classifier.keras')

    dl_results_df = pd.DataFrame(dl_results)
    dl_results_df.to_csv(DIRS['tables'] / 'table_12_deep_learning_results.csv', index=False)
    dl_results_df.to_excel(DIRS['tables'] / 'table_12_deep_learning_results.xlsx', index=False)
    display(dl_results_df)

    # Training history figures
    def plot_history(hist, title, filename):
        plt.figure(figsize=(8,5))
        plt.plot(hist.history['loss'], label='train_loss')
        plt.plot(hist.history['val_loss'], label='val_loss')
        plt.title(title)
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        save_fig(filename)
    plot_history(hist_mlp_metal, 'MLP Metal Classifier Training History', 'figure_07_mlp_metal_training_history.png')
    plot_history(hist_mlp_status, 'MLP Direct Status Baseline Training History', 'figure_08_mlp_status_training_history.png')
    plot_history(hist_mlp_reg, 'MLP Concentration Regressor Training History', 'figure_09_mlp_regression_training_history.png')
else:
    print('Deep learning skipped because TensorFlow/GPU setup is unavailable or disabled.')

In [ ]:
# ================================================================
# 14. FINAL PAPER FIGURES: CONFUSION MATRIX, REGRESSION, PIPELINE STATUS
# ================================================================

# Metal confusion matrix
cm_metal = confusion_matrix(ym_test, pred_metal_enc)
plt.figure(figsize=(6, 5))
plt.imshow(cm_metal, interpolation='nearest')
plt.title(f'Confusion Matrix - Metal Classification ({best_metal_name})')
plt.colorbar()
plt.xticks(range(len(metal_le.classes_)), metal_le.classes_, rotation=45)
plt.yticks(range(len(metal_le.classes_)), metal_le.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
for i in range(cm_metal.shape[0]):
    for j in range(cm_metal.shape[1]):
        plt.text(j, i, cm_metal[i, j], ha='center', va='center')
save_fig('figure_10_confusion_matrix_metal_best.png')

# Pipeline status confusion matrix
status_labels_sorted = sorted(pd.unique(np.concatenate([true_status_label, np.array(pred_status_label)])))
cm_status = confusion_matrix(true_status_label, pred_status_label, labels=status_labels_sorted)
plt.figure(figsize=(7, 6))
plt.imshow(cm_status, interpolation='nearest')
plt.title('Confusion Matrix - Digital Twin Status Pipeline')
plt.colorbar()
plt.xticks(range(len(status_labels_sorted)), status_labels_sorted, rotation=45, ha='right')
plt.yticks(range(len(status_labels_sorted)), status_labels_sorted)
plt.xlabel('Predicted')
plt.ylabel('Actual')
for i in range(cm_status.shape[0]):
    for j in range(cm_status.shape[1]):
        plt.text(j, i, cm_status[i, j], ha='center', va='center')
save_fig('figure_11_confusion_matrix_status_pipeline.png')

# Safety confusion matrix
safety_labels_sorted = sorted(pd.unique(np.concatenate([true_safety_label, np.array(pred_safety_label)])))
cm_safety = confusion_matrix(true_safety_label, pred_safety_label, labels=safety_labels_sorted)
plt.figure(figsize=(5, 4))
plt.imshow(cm_safety, interpolation='nearest')
plt.title('Confusion Matrix - Digital Twin Safety Pipeline')
plt.colorbar()
plt.xticks(range(len(safety_labels_sorted)), safety_labels_sorted)
plt.yticks(range(len(safety_labels_sorted)), safety_labels_sorted)
plt.xlabel('Predicted')
plt.ylabel('Actual')
for i in range(cm_safety.shape[0]):
    for j in range(cm_safety.shape[1]):
        plt.text(j, i, cm_safety[i, j], ha='center', va='center')
save_fig('figure_12_confusion_matrix_safety_pipeline.png')

# Regression actual vs predicted
plt.figure(figsize=(6, 6))
plt.scatter(true_concentration, pred_concentration, alpha=0.65, s=20)
minv = min(true_concentration.min(), pred_concentration.min())
maxv = max(true_concentration.max(), pred_concentration.max())
plt.plot([minv, maxv], [minv, maxv], linestyle='--')
plt.xlabel('Actual concentration (mg/L)')
plt.ylabel('Predicted concentration (mg/L)')
plt.title(f'Actual vs Predicted Concentration ({best_reg_name})')
save_fig('figure_13_actual_vs_predicted_concentration.png')

# Residuals
residuals = true_concentration - pred_concentration
plt.figure(figsize=(7, 5))
plt.scatter(pred_concentration, residuals, alpha=0.65, s=20)
plt.axhline(0, linestyle='--')
plt.xlabel('Predicted concentration (mg/L)')
plt.ylabel('Residual (actual - predicted)')
plt.title('Regression Residual Plot')
save_fig('figure_14_regression_residuals.png')

# Model comparison plots
plt.figure(figsize=(9, 5))
reg_results_df.set_index('model')['R2'].sort_values().plot(kind='barh')
plt.xlabel('R²')
plt.title('Concentration Regression Model Comparison')
save_fig('figure_15_regression_model_comparison.png')

plt.figure(figsize=(9, 5))
metal_results_df.set_index('model')['macro_f1'].sort_values().plot(kind='barh')
plt.xlabel('Macro F1-score')
plt.title('Metal Classification Model Comparison')
save_fig('figure_16_metal_model_comparison.png')

In [ ]:
# ================================================================
# 15. INTERPRETABILITY: PERMUTATION IMPORTANCE AND OPTIONAL SHAP
# ================================================================

# Permutation importance for best concentration regressor
print('Computing permutation importance for concentration regressor...')
perm_reg = permutation_importance(best_reg_model, X_test, yc_test, n_repeats=15, random_state=SEED, n_jobs=-1, scoring='r2')
perm_reg_df = pd.DataFrame({
    'feature': feature_cols,
    'importance_mean': perm_reg.importances_mean,
    'importance_std': perm_reg.importances_std
}).sort_values('importance_mean', ascending=False)
perm_reg_df.to_csv(DIRS['tables'] / 'table_13_permutation_importance_concentration.csv', index=False)
perm_reg_df.to_excel(DIRS['tables'] / 'table_13_permutation_importance_concentration.xlsx', index=False)
display(perm_reg_df.head(20))

plt.figure(figsize=(8, 6))
perm_reg_df.head(15).sort_values('importance_mean').set_index('feature')['importance_mean'].plot(kind='barh')
plt.xlabel('Permutation importance mean decrease in R²')
plt.title('Feature Importance for Concentration Prediction')
save_fig('figure_17_permutation_importance_concentration.png')

# Permutation importance for metal classifier
print('Computing permutation importance for metal classifier...')
perm_metal = permutation_importance(best_metal_model, X_test, ym_test, n_repeats=15, random_state=SEED, n_jobs=-1, scoring='f1_macro')
perm_metal_df = pd.DataFrame({
    'feature': feature_cols,
    'importance_mean': perm_metal.importances_mean,
    'importance_std': perm_metal.importances_std
}).sort_values('importance_mean', ascending=False)
perm_metal_df.to_csv(DIRS['tables'] / 'table_14_permutation_importance_metal.csv', index=False)
perm_metal_df.to_excel(DIRS['tables'] / 'table_14_permutation_importance_metal.xlsx', index=False)
display(perm_metal_df.head(20))

plt.figure(figsize=(8, 6))
perm_metal_df.head(15).sort_values('importance_mean').set_index('feature')['importance_mean'].plot(kind='barh')
plt.xlabel('Permutation importance mean decrease in macro F1')
plt.title('Feature Importance for Metal Classification')
save_fig('figure_18_permutation_importance_metal.png')

# Optional SHAP for tree-based model. This may be slower; limited to 300 samples.
RUN_SHAP = HAS_SHAP
if RUN_SHAP:
    try:
        # SHAP support works best on the underlying tree model after preprocessing.
        X_test_transformed = best_reg_model.named_steps['prep'].transform(X_test)
        underlying = best_reg_model.named_steps['model']
        sample_n = min(300, X_test_transformed.shape[0])
        explainer = shap.Explainer(underlying, X_test_transformed[:sample_n])
        shap_values = explainer(X_test_transformed[:sample_n])
        shap_abs = np.abs(shap_values.values).mean(axis=0)
        shap_df = pd.DataFrame({'feature': feature_cols, 'mean_abs_shap': shap_abs}).sort_values('mean_abs_shap', ascending=False)
        shap_df.to_csv(DIRS['tables'] / 'table_15_shap_importance_concentration.csv', index=False)
        shap_df.to_excel(DIRS['tables'] / 'table_15_shap_importance_concentration.xlsx', index=False)
        display(shap_df.head(20))
        plt.figure(figsize=(8, 6))
        shap_df.head(15).sort_values('mean_abs_shap').set_index('feature')['mean_abs_shap'].plot(kind='barh')
        plt.xlabel('Mean |SHAP value|')
        plt.title('SHAP Feature Importance for Concentration Prediction')
        save_fig('figure_19_shap_importance_concentration.png')
    except Exception as e:
        print('SHAP skipped due to compatibility issue:', e)

In [ ]:
# ================================================================
# 16. DIGITAL TWIN SIMULATION: SCENARIO, SENSITIVITY, MONTE CARLO
# ================================================================

# A digital twin simulation changes selected input features and observes predicted concentration/status.
# This supports dashboard visualization and paper figures.

baseline_sample = X_test.iloc[[0]].copy()

def digital_twin_predict(sample_df):
    metal_enc = best_metal_model.predict(sample_df)
    metal_label = metal_le.inverse_transform(metal_enc)
    conc_pred = np.clip(best_reg_model.predict(sample_df), 0, None)
    rows = []
    for i, (m, c) in enumerate(zip(metal_label, conc_pred)):
        s, safe, ratio, thr = status_from_metal_concentration(m, c)
        rows.append({
            'predicted_metal': m,
            'predicted_concentration_mg_L': float(c),
            'regulatory_threshold_mg_L': float(thr),
            'predicted_exceedance_ratio': float(ratio),
            'predicted_status': s,
            'predicted_safety': safe,
        })
    return pd.DataFrame(rows)

# Sensitivity over pH
ph_values = np.linspace(max(3.5, fe['pH'].quantile(0.01)), min(10.5, fe['pH'].quantile(0.99)), 80)
ph_rows = []
for ph in ph_values:
    s = baseline_sample.copy()
    s['pH'] = ph
    if 'Temp_pH_interaction' in s.columns:
        s['Temp_pH_interaction'] = s['Temperature_C'] * s['pH']
    if 'Conductivity_pH_ratio' in s.columns:
        s['Conductivity_pH_ratio'] = s['Conductivity_uS_cm'] / (s['pH'].abs() + EPS)
    pred = digital_twin_predict(s).iloc[0].to_dict()
    pred['scenario_pH'] = ph
    ph_rows.append(pred)
ph_sensitivity_df = pd.DataFrame(ph_rows)
ph_sensitivity_df.to_csv(DIRS['tables'] / 'table_16_digital_twin_ph_sensitivity.csv', index=False)
ph_sensitivity_df.to_excel(DIRS['tables'] / 'table_16_digital_twin_ph_sensitivity.xlsx', index=False)

plt.figure(figsize=(8, 5))
plt.plot(ph_sensitivity_df['scenario_pH'], ph_sensitivity_df['predicted_concentration_mg_L'])
plt.xlabel('Scenario pH')
plt.ylabel('Predicted concentration (mg/L)')
plt.title('Digital Twin pH Sensitivity Simulation')
save_fig('figure_20_digital_twin_ph_sensitivity.png')

# Monte Carlo uncertainty simulation around a baseline sample
rng = np.random.default_rng(SEED)
mc_n = 1000
mc_samples = pd.concat([baseline_sample] * mc_n, ignore_index=True)
# Perturb selected continuous features by small noise based on dataset std
perturb_cols = ['Magnitude_ohm', 'Real_Impedance_ohm', 'Imag_Impedance_ohm', 'pH', 'Temperature_C', 'Conductivity_uS_cm']
for col in perturb_cols:
    if col in mc_samples.columns:
        std = fe[col].std()
        mc_samples[col] = mc_samples[col] + rng.normal(0, 0.03 * std, size=mc_n)
# Recalculate engineered features after perturbation where possible
if {'Real_Impedance_ohm','Imag_Impedance_ohm'}.issubset(mc_samples.columns):
    mc_samples['Real_Impedance_abs'] = mc_samples['Real_Impedance_ohm'].abs()
    mc_samples['Imag_Impedance_abs'] = mc_samples['Imag_Impedance_ohm'].abs()
    mc_samples['Impedance_real_imag_ratio'] = mc_samples['Real_Impedance_abs'] / (mc_samples['Imag_Impedance_abs'] + EPS)
if 'Magnitude_ohm' in mc_samples.columns:
    mc_samples['Impedance_magnitude_log1p'] = np.log1p(mc_samples['Magnitude_ohm'].clip(lower=0))
    mc_samples['Conductance_proxy'] = 1.0 / (mc_samples['Magnitude_ohm'].abs() + EPS)
if 'Frequency_Hz' in mc_samples.columns:
    mc_samples['Frequency_log10'] = np.log10(mc_samples['Frequency_Hz'].clip(lower=EPS))
if 'Charge_Transfer_Resistance_ohm' in mc_samples.columns:
    mc_samples['Charge_Transfer_Resistance_log1p'] = np.log1p(mc_samples['Charge_Transfer_Resistance_ohm'].clip(lower=0))
if 'Double_Layer_Capacitance_F' in mc_samples.columns:
    mc_samples['Capacitance_log10_abs'] = np.log10(mc_samples['Double_Layer_Capacitance_F'].abs() + EPS)
if 'Phase_deg' in mc_samples.columns:
    mc_samples['Phase_rad'] = np.deg2rad(mc_samples['Phase_deg'])
    mc_samples['Phase_sin'] = np.sin(mc_samples['Phase_rad'])
    mc_samples['Phase_cos'] = np.cos(mc_samples['Phase_rad'])
if {'Temperature_C','pH'}.issubset(mc_samples.columns):
    mc_samples['Temp_pH_interaction'] = mc_samples['Temperature_C'] * mc_samples['pH']
if {'Conductivity_uS_cm','pH'}.issubset(mc_samples.columns):
    mc_samples['Conductivity_pH_ratio'] = mc_samples['Conductivity_uS_cm'] / (mc_samples['pH'].abs() + EPS)
if {'Charge_Transfer_Resistance_ohm','Double_Layer_Capacitance_F'}.issubset(mc_samples.columns):
    mc_samples['Rct_Cdl_product'] = mc_samples['Charge_Transfer_Resistance_ohm'] * mc_samples['Double_Layer_Capacitance_F']
if {'Charge_Transfer_Resistance_ohm','Magnitude_ohm'}.issubset(mc_samples.columns):
    mc_samples['Normalized_Rct_by_Z'] = mc_samples['Charge_Transfer_Resistance_ohm'] / (mc_samples['Magnitude_ohm'].abs() + EPS)

mc_samples = mc_samples[feature_cols]
mc_pred = digital_twin_predict(mc_samples)
mc_summary = mc_pred.describe(include='all')
mc_pred.to_csv(DIRS['tables'] / 'table_17_digital_twin_monte_carlo_predictions.csv', index=False)
mc_summary.to_csv(DIRS['tables'] / 'table_18_digital_twin_monte_carlo_summary.csv')
mc_summary.to_excel(DIRS['tables'] / 'table_18_digital_twin_monte_carlo_summary.xlsx')

plt.figure(figsize=(8, 5))
plt.hist(mc_pred['predicted_concentration_mg_L'], bins=40, alpha=0.8)
plt.xlabel('Predicted concentration (mg/L)')
plt.ylabel('Frequency')
plt.title('Digital Twin Monte Carlo Uncertainty Simulation')
save_fig('figure_21_digital_twin_monte_carlo_uncertainty.png')

In [ ]:
# ================================================================
# 17. CONSOLIDATED EXCEL TABLES FOR PAPER
# ================================================================

# Consolidate all table CSV/XLSX-like DataFrames into one Excel workbook.
consolidated_path = DIRS['tables'] / 'all_paper_tables_consolidated.xlsx'
with pd.ExcelWriter(consolidated_path, engine='openpyxl') as writer:
    threshold_df.to_excel(writer, sheet_name='00_thresholds', index=False)
    summary_stats.to_excel(writer, sheet_name='01_summary_stats')
    missing_table.to_excel(writer, sheet_name='02_missing_values')
    metal_status_table.to_excel(writer, sheet_name='03_metal_status')
    if 'metal_classification_cv' in cv_tables:
        cv_tables['metal_classification_cv'].to_excel(writer, sheet_name='04_cv_metal', index=False)
    if 'direct_status_classification_cv' in cv_tables:
        cv_tables['direct_status_classification_cv'].to_excel(writer, sheet_name='05_cv_status_direct', index=False)
    if 'concentration_regression_cv' in cv_tables:
        cv_tables['concentration_regression_cv'].to_excel(writer, sheet_name='06_cv_regression', index=False)
    metal_results_df.to_excel(writer, sheet_name='07_test_metal', index=False)
    status_results_df.to_excel(writer, sheet_name='08_test_status_direct', index=False)
    reg_results_df.to_excel(writer, sheet_name='09_test_regression', index=False)
    pipeline_metrics.to_excel(writer, sheet_name='10_pipeline_status', index=False)
    pred_table.head(100).to_excel(writer, sheet_name='11_sample_predictions', index=False)
    perm_reg_df.to_excel(writer, sheet_name='13_perm_regression', index=False)
    perm_metal_df.to_excel(writer, sheet_name='14_perm_metal', index=False)
    ph_sensitivity_df.to_excel(writer, sheet_name='16_ph_sensitivity', index=False)
    mc_summary.to_excel(writer, sheet_name='18_mc_summary')

print('Consolidated paper tables saved to:', consolidated_path)

In [ ]:
# ================================================================
# 18. REPRODUCIBILITY METADATA AND MODEL ARTIFACT EXPORT
# ================================================================

# Model bundle metadata for scientific reproducibility
model_card = {
    'project_name': PROJECT_NAME,
    'run_timestamp': RUN_TIMESTAMP,
    'dataset_source': 'Kaggle: colabsss/electrochemical-heavy-metal-sensor-data',
    'threshold_standard': THRESHOLD_STANDARD,
    'selected_thresholds_mg_L': selected_thresholds,
    'feature_columns': feature_cols,
    'best_metal_classifier': best_metal_name,
    'best_concentration_regressor': best_reg_name,
    'best_direct_status_baseline': best_status_direct_name,
    'test_metrics': {
        'metal_classification': metal_results_df.iloc[0].to_dict(),
        'concentration_regression': reg_results_df.iloc[0].to_dict(),
        'digital_twin_pipeline': pipeline_metrics.to_dict(orient='records')
    },
    'status_rule': schema['status_rule'],
    'environment_versions': versions,
    'notes_for_manuscript': [
        'Primary contamination status is computed from predicted concentration and metal-specific regulatory/literature threshold.',
        'Direct status classifiers are reported only as baselines.',
        'Use the exported schemas, model metadata, tables, and figures for reproducible manuscript reporting.'
    ]
}
with open(DIRS['models'] / 'model_card.json', 'w', encoding='utf-8') as f:
    json.dump(model_card, f, indent=2)

# Threshold JSON for reproducibility
with open(DIRS['models'] / 'regulatory_thresholds.json', 'w', encoding='utf-8') as f:
    json.dump({
        'selected_standard': THRESHOLD_STANDARD,
        'thresholds_mg_L': selected_thresholds,
        'all_threshold_records': threshold_records,
        'status_rule': schema['status_rule']
    }, f, indent=2)

# Sample feature vector and expected digital twin output for documentation and independent verification
sample_payload = {col: float(X_test.iloc[0][col]) for col in feature_cols}
with open(DIRS['models'] / 'sample_payload.json', 'w', encoding='utf-8') as f:
    json.dump(sample_payload, f, indent=2)

sample_response = digital_twin_predict(X_test.iloc[[0]]).iloc[0].to_dict()
with open(DIRS['models'] / 'sample_response.json', 'w', encoding='utf-8') as f:
    json.dump(sample_response, f, indent=2)

print('Reproducibility metadata saved in:', DIRS['models'])


In [ ]:
# ================================================================
# 19. MANUSCRIPT-READY RESULTS SUMMARY
# ================================================================

summary_lines = []
summary_lines.append('# Manuscript Results Summary')
summary_lines.append('')
summary_lines.append(f'Run timestamp: {RUN_TIMESTAMP}')
summary_lines.append(f'Threshold standard: {THRESHOLD_STANDARD}')
summary_lines.append('')
summary_lines.append('## Best Test Results')
summary_lines.append(f'- Best metal classifier: {best_metal_name}')
summary_lines.append(f"  - Accuracy: {metal_results_df.iloc[0]['accuracy']:.4f}")
summary_lines.append(f"  - Macro F1: {metal_results_df.iloc[0]['macro_f1']:.4f}")
summary_lines.append(f"  - Macro ROC-AUC OvR: {metal_results_df.iloc[0]['macro_roc_auc_ovr']:.4f}")
summary_lines.append(f'- Best concentration regressor: {best_reg_name}')
summary_lines.append(f"  - R²: {reg_results_df.iloc[0]['R2']:.4f}")
summary_lines.append(f"  - MAE: {reg_results_df.iloc[0]['MAE']:.4f} mg/L")
summary_lines.append(f"  - RMSE: {reg_results_df.iloc[0]['RMSE']:.4f} mg/L")
summary_lines.append('- Digital twin regulatory status pipeline: predicted metal + predicted concentration')
summary_lines.append(f"  - Status accuracy: {pipeline_status_acc:.4f}")
summary_lines.append(f"  - Status macro F1: {pipeline_status_f1:.4f}")
summary_lines.append(f"  - Safety accuracy: {pipeline_safety_acc:.4f}")
summary_lines.append(f"  - Safety macro F1: {pipeline_safety_f1:.4f}")
summary_lines.append('')
summary_lines.append('## Recommended wording')
summary_lines.append('The digital twin framework estimates heavy metal concentration from electrochemical sensor-response features and converts the predicted concentration into contamination status using metal-specific guideline/action thresholds. Direct status classification is reported only as a baseline, whereas the primary decision pipeline follows a physically interpretable concentration-to-threshold logic.')

summary_text = '\n'.join(summary_lines)
(DIRS['reports'] / 'manuscript_results_summary.md').write_text(summary_text, encoding='utf-8')
print(summary_text)

In [ ]:
# ================================================================
# 20. ZIP ALL OUTPUTS FOR DOWNLOAD / ARCHIVE
# ================================================================

zip_base = DIRS['zip'] / f'{PROJECT_NAME}_outputs_{RUN_TIMESTAMP}'
# Zip the whole root folder, excluding the zip folder itself to avoid recursion.
temp_export_root = BASE_DIR.parent / f'{PROJECT_NAME}_export_temp_{RUN_TIMESTAMP}'
if temp_export_root.exists():
    shutil.rmtree(temp_export_root)
shutil.copytree(BASE_DIR, temp_export_root, ignore=shutil.ignore_patterns('09_zip_export'))
zip_path = shutil.make_archive(str(zip_base), 'zip', root_dir=temp_export_root)
shutil.rmtree(temp_export_root)

print('ZIP output saved to:', zip_path)
if IN_COLAB:
    print('You can download it from Google Drive path above, or use files.download if needed.')
    # Uncomment if you want browser download directly:
    # from google.colab import files
    # files.download(zip_path)